In [1]:
import duckdb
import pandas as pd
import numpy as np

In [5]:
# lọc ra các dòng có tỉnh Bình Dương

duckdb.query("""
    COPY (
        SELECT * 
        FROM '../datas/vietnam_real_estate.csv' 
        WHERE lower(province_name) LIKE '%bình dương%' 
           OR lower(province_name) LIKE '%binh duong%'
    ) TO '../datas/binh_duong_real_estates.csv' (HEADER, DELIMITER ',')
""")

# DATA PROFILING

In [ ]:
file_path = '../datas/binh_duong_real_estates.csv'
df = pd.read_csv(file_path)

total_rows, total_cols = df.shape
print(f"=== TỔNG QUAN TẬP DỮ LIỆU: {total_rows:,} dòng | {total_cols} cột ===\n")

# ==============================================================================
# a. KIỂM TRA DỮ LIỆU THIẾU (MISSING VALUES)
# ==============================================================================
print("-" * 60)
print("a. BÁO CÁO DỮ LIỆU THIẾU (MISSING VALUES)")
print("-" * 60)
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Rate (%)': (df.isnull().sum() / total_rows * 100).round(2)
}).sort_values(by='Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0].to_string())
print()

# ==============================================================================
# b. KIỂM TRA DỮ LIỆU TRÙNG LẶP (DUPLICATES)
# ==============================================================================
print("-" * 60)
print("b. BÁO CÁO DỮ LIỆU TRÙNG LẶP (DUPLICATES)")
print("-" * 60)
exact_dups = df.duplicated().sum()
print(f"- Trùng lặp tuyệt đối (Exact Duplicates): {exact_dups:,} bản ghi ({exact_dups/total_rows*100:.2f}%)")

biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
valid_biz_cols = [c for c in biz_cols if c in df.columns]
biz_dups = df.duplicated(subset=valid_biz_cols, keep='first').sum()
print(f"- Trùng lặp nghiệp vụ (Business Duplicates): {biz_dups:,} bản ghi ({biz_dups/total_rows*100:.2f}%)\n")

# ==============================================================================
# c. KIỂM TRA DỮ LIỆU KHÔNG HỢP LỆ & ĐỊNH DẠNG (INVALID & FORMAT VALUES)
# ==============================================================================
print("-" * 60)
print("c. BÁO CÁO DỮ LIỆU KHÔNG HỢP LỆ (INVALID VALUES)")
print("-" * 60)

invalid_price = (df['price'] <= 0).sum() if 'price' in df.columns else 0
invalid_area = (df['area'] < 10).sum() if 'area' in df.columns else 0

print(f"- Số bản ghi có 'price' <= 0: {invalid_price:,}")
print(f"- Số bản ghi có 'area' < 10 m²: {invalid_area:,}")

text_cols = ['district_name', 'ward_name', 'name']
for col in text_cols:
    if col in df.columns:
        s = df[col].astype(str)
        has_space = (s != s.str.strip()).sum()
        print(f"- Cột '{col}' chứa khoảng trắng thừa ở đầu/cuối: {has_space:,} dòng")
print()

# ==============================================================================
# d. PHÂN TÍCH NGOẠI LỆ DỰA TRÊN ĐƠN GIÁ (PRICE / AREA)
# ==============================================================================
print("-" * 60)
print("d. BÁO CÁO DỮ LIỆU NGOẠI LỆ (ANOMALIES & OUTLIERS)")
print("-" * 60)

if 'price' in df.columns and 'area' in df.columns:
    df_temp = df.dropna(subset=['price', 'area']).copy()
    df_temp = df_temp[(df_temp['price'] > 0) & (df_temp['area'] >= 10)]
    
    # Profile thêm biến Đơn giá (triệu/m2 hoặc đồng/m2)
    df_temp['price_per_m2'] = df_temp['price'] / df_temp['area']
    
    Q1 = df_temp['price_per_m2'].quantile(0.25)
    Q3 = df_temp['price_per_m2'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_iqr = ((df_temp['price_per_m2'] < lower_bound) | (df_temp['price_per_m2'] > upper_bound)).sum()
    extreme_high_price = (df_temp['price'] > 500_000_000_000).sum()
    
    print(f"- Đơn giá m² (Min: {df_temp['price_per_m2'].min():,.0f} | Max: {df_temp['price_per_m2'].max():,.0f} VNĐ/m²)")
    print(f"- Số điểm ngoại lệ đơn giá theo IQR (Price/m²): {outliers_iqr:,} dòng")
    print(f"- Số bản ghi tổng giá cực đoan (> 500 tỷ VNĐ): {extreme_high_price:,} dòng")

=== TỔNG QUAN TẬP DỮ LIỆU: 42,783 dòng | 19 cột ===

------------------------------------------------------------
a. BÁO CÁO DỮ LIỆU THIẾU (MISSING VALUES)
------------------------------------------------------------
                   Missing_Count  Missing_Rate (%)
house_depth                41984             98.13
floor_count                37184             86.91
road_width                 35255             82.40
balcony_direction          34536             80.72
house_direction            27772             64.91
frontage_width             23811             55.66
project_name               23004             53.77
street_name                22874             53.47
bathroom_count             19268             45.04
bedroom_count              18285             42.74
ward_name                   3375              7.89
price                       3324              7.77

------------------------------------------------------------
b. BÁO CÁO DỮ LIỆU TRÙNG LẶP (DUPLICATES)
----------------

# DATA CLEANING

In [3]:
# 1. Đọc dữ liệu thô
file_path = '../datas/binh_duong_real_estates.csv'
df = pd.read_csv(file_path)
initial_rows, initial_cols = df.shape
print(f"=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: {initial_rows:,} dòng | {initial_cols} cột) ===\n")

# 2. [R1] Loại bỏ cột khuyết > 80%
drop_cols = ['house_depth', 'floor_count', 'road_width', 'balcony_direction']
existing_drop_cols = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=existing_drop_cols)
print(f"[R1] Đã loại bỏ {len(existing_drop_cols)} cột khuyết nặng > 80%.")

# 3. [R2 & R5] Xóa dòng khuyết price và area < 10m²
df_clean = df_clean.dropna(subset=['price'])
df_clean = df_clean[df_clean['area'] >= 10]
print(f"[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².")

# 4. [R6] Chuẩn hóa khoảng trắng chuỗi văn bản (Trim/Strip)
text_cols = ['district_name', 'ward_name', 'name']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
print(f"[R6] Đã xóa khoảng trắng thừa ở đầu/cuối chuỗi văn bản.")

# 5. [R3] Gán giá trị mặc định cho thuộc tính Dimension khuyết
impute_cols = {
    'ward_name': 'Chưa xác định',
    'house_direction': 'Chưa xác định',
    'legal_status': 'Chưa xác định'
}
for col, val in impute_cols.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(val)
print(f"[R3] Đã gán giá trị mặc định 'Chưa xác định' cho các thuộc tính Dimension.")

# 6. [R4] Khử trùng lặp nghiệp vụ (Business Deduplication)
biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
valid_biz_cols = [c for c in biz_cols if c in df_clean.columns]
df_clean = df_clean.drop_duplicates(subset=valid_biz_cols, keep='first')
print(f"[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.")

# 7. [R7] Xử lý dữ liệu ngoại lệ cực đoan (Outliers & Anomalies)
# Lọc bỏ các dòng tổng giá cực đoan > 500 tỷ VNĐ ra khỏi dataset chính
df_outliers = df_clean[df_clean['price'] > 500_000_000_000]
df_clean = df_clean[df_clean['price'] <= 500_000_000_000]
print(f"[R7] Đã lọc bỏ {len(df_outliers)} dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.")

# 8. Xuất file dữ liệu sạch
final_rows, final_cols = df_clean.shape
output_clean_path = '../datas/cleaned_binh_duong_real_estates.csv'
output_outlier_path = '../datas/outliers_binh_duong_real_estates.csv'

df_clean.to_csv(output_clean_path, index=False, encoding='utf-8-sig')
df_outliers.to_csv(output_outlier_path, index=False, encoding='utf-8-sig')

print(f"\n=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH ===")
print(f"- Số dòng sạch còn lại: {final_rows:,} dòng (Đã lọc bỏ {initial_rows - final_rows:,} dòng rác/ngoại lệ)")
print(f"- Số cột còn lại: {final_cols} cột")
print(f"- File dữ liệu sạch đã lưu tại: {output_clean_path}")
print(f"- File dữ liệu ngoại lệ đã lưu tại: {output_outlier_path}")

=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: 42,783 dòng | 19 cột) ===

[R1] Đã loại bỏ 4 cột khuyết nặng > 80%.
[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².
[R6] Đã xóa khoảng trắng thừa ở đầu/cuối chuỗi văn bản.
[R3] Đã gán giá trị mặc định 'Chưa xác định' cho các thuộc tính Dimension.
[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.
[R7] Đã lọc bỏ 17 dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.

=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH ===
- Số dòng sạch còn lại: 36,303 dòng (Đã lọc bỏ 6,480 dòng rác/ngoại lệ)
- Số cột còn lại: 15 cột
- File dữ liệu sạch đã lưu tại: ../datas/cleaned_binh_duong_real_estates.csv
- File dữ liệu ngoại lệ đã lưu tại: ../datas/outliers_binh_duong_real_estates.csv
